In [4]:
# Cell 1 — Setup
"""
05_baselines.ipynb
==================
Train DirectGNN baseline (same arch, no physics)
and compare with TGNN-Solv.
"""

import sys
sys.path.insert(0, "../src")

import torch
import pandas as pd
import matplotlib.pyplot as plt

from tgnn_solv.config import TGNNSolvConfig
from tgnn_solv.inference import load_model
from tgnn_solv.data import make_loaders, PROCESSED_DIR
from tgnn_solv.baselines import run_baseline, compare_with_tgnn

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

Device: mps


In [5]:
# Cell 2 — Load data
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

cfg = TGNNSolvConfig(
    hidden_dim=256,
    n_gnn_layers=6,
    n_cross_attn_layers=3,
    n_attn_heads=8,
    pair_dim=512,
)

train_loader, val_loader, test_loader = make_loaders(
    train_df, val_df, test_df, batch_size=cfg.batch_size,
)

  Dataset: 7,787 valid (0 dropped, 2412 cached graphs)
  Train: 104,625 samples, 1,634 batches
  Val: 7,785 samples, 122 batches
  Test: 7,787 samples, 122 batches


In [ ]:
# Cell 3 — Train DirectGNN baseline
baseline_metrics = run_baseline(
    train_loader, val_loader, test_loader,
    cfg=cfg, device=DEVICE,
    n_epochs=2, patience=20,
)

In [ ]:
# Cell 4 — Compare with TGNN-Solv
model, model_cfg = load_model("../checkpoints/tgnn_solv_trained.pt", DEVICE)

comparison = compare_with_tgnn(
    model, model_cfg, baseline_metrics,
    test_loader, test_df,
)

comparison.to_csv("../checkpoints/baseline_comparison.csv", index=False)

In [ ]:
# Cell 5 — Visualization

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = comparison["name"].values
colors = ["coral", "steelblue"]

for ax, metric, title, better in [
    (axes[0], "mae", "MAE (ln x₂) ↓", "lower"),
    (axes[1], "rmse", "RMSE (ln x₂) ↓", "lower"),
    (axes[2], "r2", "R² ↑", "higher"),
]:
    vals = comparison[metric].values
    bars = ax.bar(names, vals, color=colors, width=0.5, edgecolor="black")
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(metric.upper())

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{val:.3f}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.savefig("../checkpoints/baseline_comparison.png", dpi=150)
plt.show()

In [ ]:
# Cell 6 — Temperature encoding sanity check

from tgnn_solv.baselines.temperature import ThermometerEncoder

enc = ThermometerEncoder(n_bins=10, T_min=200, T_max=500)

test_temps = torch.tensor([250.0, 298.15, 350.0, 450.0])
encoded = enc.encode(test_temps)

print("Temperature encoding examples (10 bins, 200-500 K):")
print(f"Bin width: {enc.bin_width:.0f} K")
for i, T_val in enumerate(test_temps):
    vals = [f"{v:.2f}" for v in encoded[i].tolist()]
    print(f"  T={T_val.item():6.1f} K → [{', '.join(vals)}]")